# Bible Notely — upload para YouTube (v2)

Envia MP4s da pasta EBD no Drive para o YouTube (**não listado**), usando título `IBQB - EBD|Culto - DD/MM/AAAA - {título do estudo}` e descrição com o resumo da v1.

## Antes de rodar (uma vez)
1. No Google Cloud: ative **YouTube Data API v3** e **Google Drive API**.
2. Tela de consentimento OAuth (externo) com seu e-mail como usuário de teste.
3. Crie credenciais OAuth **Desktop** e salve o JSON como `client_secret.json` na pasta `auto-ebd-youtube` do Drive (ao lado de `catalog.json`).
4. Verifique o canal no YouTube (telefone) para vídeos com mais de 15 minutos.
5. Na sua máquina: `cli youtube-catalog` (gera `catalog.json` no Drive).
6. Depois dos uploads: `cli youtube-sync` (grava URLs no SQLite).

Cota típica: ~6 uploads/dia (10 000 unidades, 1600 por vídeo).

In [ ]:
!pip -q install google-api-python-client google-auth-oauthlib google-auth-httplib2

from google.colab import drive as colab_drive
colab_drive.mount('/content/drive')

import json
import os
import shutil
from pathlib import Path

REPO_YOUTUBE_LIB = Path('/content/drive/MyDrive/caminho/para/bible-notely/colab/youtube_lib.py')
# Ajuste o caminho acima ou copie youtube_lib.py para /content/youtube_lib.py
LOCAL_LIB = Path('/content/youtube_lib.py')
if REPO_YOUTUBE_LIB.exists():
    shutil.copy(REPO_YOUTUBE_LIB, LOCAL_LIB)
elif not LOCAL_LIB.exists():
    raise FileNotFoundError('Copie colab/youtube_lib.py para /content/youtube_lib.py')

import importlib.util
spec = importlib.util.spec_from_file_location('youtube_lib', LOCAL_LIB)
yt = importlib.util.module_from_spec(spec)
spec.loader.exec_module(yt)
print('youtube_lib carregado:', LOCAL_LIB)

In [ ]:
DRIVE_FOLDER_ID = os.environ.get('GOOGLE_DRIVE_FOLDER_ID', '')  # pasta raiz EBD
MAX_UPLOADS = int(os.environ.get('MAX_UPLOADS', '6'))
META_FOLDER_NAME = 'auto-ebd-youtube'
WORK = Path('/content/youtube_work')
WORK.mkdir(exist_ok=True)

if not DRIVE_FOLDER_ID:
    DRIVE_FOLDER_ID = input('ID da pasta raiz EBD no Drive: ').strip()

In [ ]:
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
import io

SCOPES = [
    'https://www.googleapis.com/auth/drive.readonly',
    'https://www.googleapis.com/auth/youtube.upload',
    'https://www.googleapis.com/auth/youtube',
]

def find_meta_folder(service, root_id):
    q = f"name = '{META_FOLDER_NAME}' and '{root_id}' in parents and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    res = service.files().list(q=q, fields='files(id)', pageSize=1, supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
    files = res.get('files') or []
    if not files:
        raise RuntimeError('Pasta auto-ebd-youtube não encontrada; rode youtube-catalog na máquina')
    return files[0]['id']

def download_drive_file(service, file_id, dest: Path):
    request = service.files().get_media(fileId=file_id)
    with dest.open('wb') as fh:
        downloader = MediaIoBaseDownload(fh, request, chunksize=8 * 1024 * 1024)
        done = False
        while not done:
            _, done = downloader.next_chunk()

def read_json_by_name(service, folder_id, name):
    q = f"name = '{name}' and '{folder_id}' in parents and trashed = false"
    res = service.files().list(q=q, fields='files(id)', pageSize=1, supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
    files = res.get('files') or []
    if not files:
        return None
    path = WORK / name
    download_drive_file(service, files[0]['id'], path)
    return json.loads(path.read_text(encoding='utf-8'))

def upload_json(service, folder_id, name, payload):
    path = WORK / name
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    q = f"name = '{name}' and '{folder_id}' in parents and trashed = false"
    res = service.files().list(q=q, fields='files(id)', pageSize=1, supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
    media = MediaFileUpload(str(path), mimetype='application/json', resumable=True)
    files = res.get('files') or []
    if files:
        service.files().update(fileId=files[0]['id'], media_body=media).execute()
    else:
        service.files().create(body={'name': name, 'parents': [folder_id]}, media_body=media, fields='id').execute()

meta_folder_id = None
drive_service = None
youtube = None
state = yt.empty_state()
catalog = {'items': []}

In [ ]:
# OAuth (primeira vez: siga o link; depois reutiliza token.json no Drive)
from pathlib import Path

token_path = WORK / 'token.json'
secret_path = WORK / 'client_secret.json'

creds = None
if token_path.exists():
    creds = Credentials.from_authorized_user_file(str(token_path), SCOPES)
if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        if not secret_path.exists():
            raise FileNotFoundError('Baixe client_secret.json para /content/youtube_work/')
        flow = InstalledAppFlow.from_client_secrets_file(str(secret_path), SCOPES)
        creds = flow.run_console()
    token_path.write_text(creds.to_json(), encoding='utf-8')

drive_service = build('drive', 'v3', credentials=creds)
youtube = build('youtube', 'v3', credentials=creds)
meta_folder_id = find_meta_folder(drive_service, DRIVE_FOLDER_ID)

catalog = read_json_by_name(drive_service, meta_folder_id, 'catalog.json') or {'items': []}
state_raw = read_json_by_name(drive_service, meta_folder_id, 'state.json')
state = yt.load_state(state_raw)
print('Itens no catálogo:', len(catalog.get('items') or []))
print('Uploads já registrados:', len(state.get('uploads') or {}))

In [ ]:
# Listar MP4s (mesmo padrão de pasta ANO/Mes/Dia)
PATH_RE = yt.PATH_RE
videos = []
folders = {DRIVE_FOLDER_ID: ''}
queue = [DRIVE_FOLDER_ID]
while queue:
    folder_id = queue.pop()
    page = None
    while True:
        res = drive_service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields='nextPageToken, files(id,name,mimeType,size,createdTime,parents)',
            pageSize=200,
            pageToken=page,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()
        for item in res.get('files', []):
            if item['mimeType'] == 'application/vnd.google-apps.folder':
                rel = '/'.join(p for p in (folders[folder_id], item['name']) if p)
                folders[item['id']] = rel
                queue.append(item['id'])
            elif item['name'].lower().endswith('.mp4'):
                rel = folders[folder_id]
                d = yt.parse_path_date(rel)
                videos.append({
                    'id': item['id'],
                    'name': item['name'],
                    'folder_id': folder_id,
                    'path': rel,
                    'size': int(item.get('size') or 0),
                    'createdTime': item.get('createdTime'),
                    'lecture_date': d.isoformat() if d else '',
                })
        page = res.get('nextPageToken')
        if not page:
            break
print('MP4 encontrados:', len(videos))

In [ ]:
catalog_by_id = yt.catalog_index(catalog)
rows = [yt.merge_video_row(v, catalog_by_id, state) for v in videos]
rows.sort(key=lambda r: (r.get('lecture_date') or '', r.get('video_filename') or ''), reverse=True)

for r in rows[:30]:
    print(
        r['drive_path'],
        '|', r['video_filename'],
        '|', r['playlist'],
        '|', r['youtube_title'] or '(sem título)',
        '|', r['skip_reason'] or 'OK',
    )

In [ ]:
PLAYLIST_TITLES = {
    yt.PLAYLIST_EBD: 'IBQB — EBD',
    yt.PLAYLIST_CULT_MANHA: 'IBQB — Culto da manhã',
    yt.PLAYLIST_CULT_NOITE: 'IBQB — Culto da noite',
}

def ensure_playlists(youtube_api, state):
    state.setdefault('playlists', {})
    for key, title in PLAYLIST_TITLES.items():
        if state['playlists'].get(key):
            continue
        body = {
            'snippet': {'title': title, 'description': 'Arquivo IBQB (não listado)'},
            'status': {'privacyStatus': 'unlisted'},
        }
        created = youtube_api.playlists().insert(part='snippet,status', body=body).execute()
        state['playlists'][key] = created['id']
    return state

state = ensure_playlists(youtube, state)
upload_json(drive_service, meta_folder_id, 'state.json', state)
print('Playlists:', state['playlists'])

In [ ]:
def upload_one(row):
    file_id = row['drive_file_id']
    local = WORK / row['video_filename']
    download_drive_file(drive_service, file_id, local)
    body = {
        'snippet': {
            'title': row['youtube_title'],
            'description': row['description'],
            'categoryId': '29',
        },
        'status': {
            'privacyStatus': 'unlisted',
            'selfDeclaredMadeForKids': False,
        },
    }
    media = MediaFileUpload(str(local), chunksize=8 * 1024 * 1024, resumable=True)
    request = youtube.videos().insert(part='snippet,status', body=body, media_body=media)
    response = None
    while response is None:
        status, response = request.next_chunk()
        if status:
            print(f"  upload {int(status.progress() * 100)}%")
    video_id = response['id']
    playlist_id = state['playlists'][row['playlist']]
    youtube.playlistItems().insert(
        part='snippet',
        body={'snippet': {'playlistId': playlist_id, 'resourceId': {'kind': 'youtube#video', 'videoId': video_id}}},
    ).execute()
    yt.record_upload(
        state,
        drive_file_id=file_id,
        youtube_id=video_id,
        title=row['youtube_title'],
        drive_path=row['drive_path'],
        playlist=row['playlist'],
    )
    upload_json(drive_service, meta_folder_id, 'state.json', state)
    local.unlink(missing_ok=True)
    return video_id

sent = skipped = failed = 0
for row in rows:
    if sent >= MAX_UPLOADS:
        break
    if row['skip_reason']:
        skipped += 1
        continue
    print('Enviando', row['video_filename'], '→', row['youtube_title'])
    try:
        vid = upload_one(row)
        print('  OK', yt.youtube_watch_url(vid))
        sent += 1
    except Exception as exc:
        print('  FALHA', exc)
        failed += 1
        if 'quota' in str(exc).lower() or '403' in str(exc):
            print('Cota esgotada; retome amanhã.')
            break

print(f'Resumo: enviados={sent} pulados={skipped} falhas={failed}')
print('Na máquina: PYTHONPATH=$PWD .venv/bin/python -m backend.app.cli youtube-sync')